# Amazon Review Alignment: A100 Formal Pipeline

使用 `Qwen/Qwen3.5-2B`、BF16 与 4-bit QLoRA 执行正式的五模型实验。
该配置控制了 PPO/GRPO prompt 数与评估规模，但 Colab Compute Unit
消耗是动态的，不能保证固定在 100 CU 内。


## 1. 挂载 Drive 并加载项目

本 Notebook 将仓库放在 Google Drive，训练输出和 checkpoint 会在断开
Colab 后保留。目标 GPU：`A100`。

开始前必须先将本地最新代码和本 Notebook 提交并推送到 GitHub `main`
分支，否则下方 `git clone` 会获取旧版本。


In [4]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/j156734119/Amazon-reviews-2023-electronics-SFT-DPO.git"
REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / ".git").exists():
    status = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--short"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if status:
        print("Preserving Colab-local tracked changes before pull:")
        print(status)
        subprocess.run(
            [
                "git",
                "-C",
                str(REPO_DIR),
                "stash",
                "push",
                "-m",
                "colab-auto-stash-before-pull",
            ],
            check=True,
        )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


KeyboardInterrupt: 

## 2. 安装依赖

执行后使用 Colab 菜单 **运行时 -> 重新启动会话**。重启后从下一单元格
继续，不需要再次执行安装。


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=False,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "pip",
        "setuptools",
        "wheel",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,eval,dev]"],
    check=True,
)
print("Installation complete. Restart the Colab runtime now.")


## 3. 重启后恢复目录、加载 Secrets

在 Colab 左侧钥匙图标中添加 `OPENAI_API_KEY`。`HF_TOKEN` 对公开模型
是可选的，但能提高 Hugging Face 下载限额。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
os.chdir(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
CONFIG = "configs/rlhf_a100.yaml"

try:
    import amazon_review_alignment
    import bitsandbytes
    import peft
    import transformers
    import trl
except (ImportError, ModuleNotFoundError):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            ".[train,eval,dev]",
        ],
        cwd=REPO_DIR,
        check=True,
    )
    import amazon_review_alignment
    import bitsandbytes
    import peft
    import transformers
    import trl

for secret_name in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None
    if value:
        os.environ[secret_name] = value

def cli(*arguments: str, check: bool = True) -> subprocess.CompletedProcess:
    command = [
        sys.executable,
        "-m",
        "amazon_review_alignment.cli",
        *(str(argument) for argument in arguments),
    ]
    print("\n$", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    output_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        output_lines.append(line)
    returncode = process.wait()
    result = subprocess.CompletedProcess(
        command,
        returncode,
        stdout="".join(output_lines),
        stderr=None,
    )
    if check and returncode:
        raise RuntimeError(
            f"Command failed with exit code {returncode}: "
            + " ".join(command)
        )
    return result

print("Config:", CONFIG)
print("Package:", Path(amazon_review_alignment.__file__).resolve())
print(
    "Training stack:",
    transformers.__version__,
    trl.__version__,
    peft.__version__,
    bitsandbytes.__version__,
)
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("HF token loaded:", bool(os.getenv("HF_TOKEN")))


## A100 环境检查


In [ ]:
import importlib.metadata

import torch
import transformers
import trl

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a Colab GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name)
print(f"VRAM: {total_gib:.2f} GiB")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", importlib.metadata.version("peft"))

if "A100".lower() not in gpu_name.lower():
    raise RuntimeError("Expected A100, but Colab assigned: " + gpu_name)
if total_gib < 38:
    raise RuntimeError("Insufficient GPU memory for this profile.")
if True and not torch.cuda.is_bf16_supported():
    raise RuntimeError("This profile requires BF16 support.")


## 4. 选择 A100 Mini Smoke 或正式训练

首次运行保持 `RUN_MODE="mini"`。它使用 Qwen3.5-2B 和真实 A100
装配，但只训练一步，并使用 60 条评论验证完整链路。全部通过后改成
`RUN_MODE="formal"`，重新从数据准备开始执行正式实验。

Mini 与正式输出目录完全隔离，不会相互复用 checkpoint。


In [ ]:
import yaml

from amazon_review_alignment.config import load_config

RUN_MODE = "formal"  # mini | formal

if RUN_MODE == "mini":
    merged = load_config(REPO_DIR / "configs" / "rlhf_a100.yaml")
    merged.pop("_config_path", None)

    old_root = "outputs/a100-qwen3.5-2b"
    new_root = "outputs/a100-mini-qwen3.5-2b"

    def replace_output_paths(value):
        if isinstance(value, dict):
            return {
                key: replace_output_paths(item)
                for key, item in value.items()
            }
        if isinstance(value, list):
            return [replace_output_paths(item) for item in value]
        if isinstance(value, str):
            return value.replace(old_root, new_root)
        return value

    merged = replace_output_paths(merged)
    merged["project"]["output_dir"] = new_root
    merged["data"].update(
        {
            "sample_size": 60,
            "max_scanned_reviews": 20000,
            "rating_targets": {
                "1": 12,
                "2": 12,
                "3": 12,
                "4": 12,
                "5": 12,
            },
            "splits": {
                "train": 42,
                "validation": 6,
                "test": 12,
            },
        }
    )
    merged["teacher"].update(
        {
            "pilot_size": 5,
            "max_estimated_cost_usd": 1.0,
        }
    )
    merged["training"]["sft"]["max_steps"] = 1
    merged["training"]["dpo"]["max_steps"] = 1
    merged["rlhf"].update(
        {
            "human_calibration_samples": 0,
            "ai_reward_train_pairs": 4,
            "ai_reward_validation_pairs": 2,
            "ppo_prompt_count": 4,
        }
    )
    merged["rlhf"]["reward"]["max_steps"] = 1
    merged["rlhf"]["ppo"]["total_episodes"] = 4
    merged["rlhf"]["ppo"]["gradient_accumulation_steps"] = 1
    merged["rlhf"]["ppo"]["save_steps"] = 1
    merged["rlhf"]["grpo"]["prompt_count"] = 4
    merged["rlhf"]["grpo"]["max_steps"] = 1
    merged["evaluation"]["max_test_samples"] = 4

    mini_path = Path("/content/rlhf_a100_mini.yaml")
    mini_path.write_text(
        yaml.safe_dump(merged, sort_keys=False),
        encoding="utf-8",
    )
    CONFIG = str(mini_path)
    effective = merged
elif RUN_MODE == "formal":
    CONFIG = "configs/rlhf_a100.yaml"
    effective = load_config(REPO_DIR / CONFIG)
else:
    raise ValueError("RUN_MODE must be 'mini' or 'formal'.")

print("Run mode:", RUN_MODE)
print("Effective config:", CONFIG)
print(
    "PPO auxiliary models in 4-bit:",
    effective["rlhf"]["ppo"]["auxiliary_model_load_in_4bit"],
)


## 4. 测试、准备数据并生成 Base baseline


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pytest"],
    cwd=REPO_DIR,
    check=True,
)
cli("prepare-data", "--config", CONFIG)


In [ ]:
cli(
    "evaluate",
    "--config",
    CONFIG,
    "--variants",
    "base",
    "--force-inference",
)

import pandas as pd
from amazon_review_alignment.config import load_config

output_root = Path(
    load_config(CONFIG)["project"]["output_dir"]
)
display(pd.read_csv(output_root / "evaluation" / "metrics.csv"))


## 5. 教师数据

Pilot 会立即调用 OpenAI API。Batch 提交后可能需要等待；提交成功后可以
关闭 GPU Runtime，稍后重新连接并重复“检查 Batch”单元格。


In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets first.")
cli("teacher-pilot", "--config", CONFIG)


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive, userdata

drive.mount("/content/drive")

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
CONFIG = "configs/rlhf_a100.yaml"

# 进入项目目录，保证相对配置路径有效
os.chdir(REPO_DIR)

# 加载 Colab Secrets
for secret_name in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None

    if value:
        os.environ[secret_name] = value

# 重新安装当前项目源码
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(REPO_DIR),
    ],
    check=True,
)

def cli(*arguments, check=True):
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(argument) for argument in arguments),
    ]

    print("\n$", " ".join(command), flush=True)

    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)

    return_code = process.wait()

    if check and return_code:
        print("\n===== LAST 200 LINES =====")
        print("".join(lines[-200:]))
        raise RuntimeError(
            f"Command failed with exit code {return_code}"
        )

    return subprocess.CompletedProcess(
        command,
        return_code,
        stdout="".join(lines),
    )

print("Repository:", REPO_DIR)
print("Config:", CONFIG)
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("HF token loaded:", bool(os.getenv("HF_TOKEN")))

In [ ]:
# 第一次执行会提交 Batch；后续重复执行会查询并下载结果。
cli("teacher-batch", "--config", CONFIG)


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
CONFIG = str(REPO_DIR / "configs" / "rlhf_a100.yaml")

if not REPO_DIR.exists():
    raise RuntimeError(f"项目目录不存在：{REPO_DIR}")

# 加载 Secrets
for name in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        value = userdata.get(name)
    except Exception:
        value = None

    if value:
        os.environ[name] = value

# 重新安装本地项目
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(REPO_DIR),
    ],
    check=True,
)

# 确保当前 Python 进程能立即找到 src package
src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.chdir(REPO_DIR)

from amazon_review_alignment.config import load_config

def cli(*arguments, check=True):
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(arg) for arg in arguments),
    ]

    print("\n$", " ".join(command), flush=True)

    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)

    return_code = process.wait()

    if check and return_code:
        print("".join(lines[-200:]))
        raise RuntimeError(
            f"Command failed with exit code {return_code}"
        )

    return subprocess.CompletedProcess(
        command,
        return_code,
        stdout="".join(lines),
    )

print("Repository:", REPO_DIR)
print("Config:", CONFIG)
print("Package loaded successfully")
print("OpenAI key:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
from pathlib import Path

from amazon_review_alignment.config import load_config

effective_config = load_config(CONFIG)
output_root = (
    REPO_DIR / effective_config["project"]["output_dir"]
).resolve()

train_preferences = (
    output_root / "teacher" / "preferences_train.jsonl"
)
validation_preferences = (
    output_root / "teacher" / "preferences_validation.jsonl"
)

print("Output root:", output_root)
print("Train file exists:", train_preferences.exists())
print(
    "Validation file exists:",
    validation_preferences.exists(),
)

# 文件不存在时，查询已有 Batch 并尝试下载结果
if not train_preferences.exists() or not validation_preferences.exists():
    cli("teacher-batch", "--config", CONFIG)

if not train_preferences.exists() or not validation_preferences.exists():
    raise RuntimeError(
        "OpenAI Batch 尚未完成。可以断开 A100，稍后重新运行本单元格。"
    )

train_rows = sum(
    1 for line in train_preferences.open(encoding="utf-8")
    if line.strip()
)
validation_rows = sum(
    1 for line in validation_preferences.open(encoding="utf-8")
    if line.strip()
)

print("Teacher train rows:", train_rows)
print("Teacher validation rows:", validation_rows)

if train_rows == 0 or validation_rows == 0:
    raise RuntimeError("Teacher 数据文件存在，但没有有效记录。")

print("Teacher data ready. 可以开始 SFT。")

## 6. SFT、合并权重与 DPO


In [ ]:
cli("train-sft", "--config", CONFIG)


In [ ]:
cli("merge-sft", "--config", CONFIG)


In [ ]:
cli("train-dpo", "--config", CONFIG)


In [ ]:
cli(
    "evaluate",
    "--config",
    CONFIG,
    "--variants",
    "base",
    "sft",
    "dpo",
)
display(pd.read_csv(output_root / "evaluation" / "metrics.csv"))


In [ ]:
from pathlib import Path
import pandas as pd

output_root = (
    REPO_DIR
    / "outputs"
    / "a100-qwen3.5-2b"
)

for name in ("sft", "sft-merged", "dpo"):
    path = output_root / "models" / name
    print(name, path.exists(), path)

for variant in ("base", "sft", "dpo"):
    path = (
        output_root
        / "evaluation"
        / "predictions"
        / f"{variant}.jsonl"
    )

    rows = 0
    if path.exists():
        rows = sum(
            1 for line in path.open(encoding="utf-8")
            if line.strip()
        )

    print(variant, "predictions:", rows)

metrics_path = output_root / "evaluation" / "metrics.csv"
display(pd.read_csv(metrics_path))

In [ ]:
from pathlib import Path
import json
import pandas as pd

output_root = (
    REPO_DIR
    / "outputs"
    / "a100-qwen3.5-2b"
)
prediction_dir = output_root / "evaluation" / "predictions"

def read_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]

sft_rows = read_jsonl(prediction_dir / "sft.jsonl")
dpo_rows = read_jsonl(prediction_dir / "dpo.jsonl")

metrics = pd.read_csv(
    output_root / "evaluation" / "metrics.csv"
)
display(metrics)

print("SFT rows:", len(sft_rows))
print("DPO rows:", len(dpo_rows))

# 展示前 10 个 DPO 输出
for index, (sft, dpo) in enumerate(
    zip(sft_rows, dpo_rows),
    start=1,
):
    if index > 10:
        break

    print("\n" + "=" * 100)
    print("INDEX:", index)
    print("REVIEW:", dpo["text"][:800])
    print("\nSFT:")
    print(sft["raw_output"])
    print("\nDPO:")
    print(dpo["raw_output"])

In [ ]:
from pathlib import Path
import json

dpo_dir = output_root / "models" / "dpo"
sft_merged_dir = output_root / "models" / "sft-merged"

print("DPO directory:", dpo_dir)
print("DPO exists:", dpo_dir.exists())
print("Merged SFT exists:", sft_merged_dir.exists())

for path in sorted(dpo_dir.iterdir()):
    print(path.name, path.stat().st_size)

adapter_config = dpo_dir / "adapter_config.json"

if adapter_config.exists():
    print("\nDPO adapter config:")
    print(
        json.dumps(
            json.loads(
                adapter_config.read_text(encoding="utf-8")
            ),
            indent=2,
        )
    )

In [ ]:
from pathlib import Path
import json
import pandas as pd

state_path = (
    output_root
    / "models"
    / "dpo"
    / "checkpoint-65"
    / "trainer_state.json"
)

state = json.loads(state_path.read_text(encoding="utf-8"))
history = pd.DataFrame(state["log_history"])

display(history)
history.to_csv(
    output_root / "evaluation" / "dpo_training_history.csv",
    index=False,
)

In [ ]:
from pathlib import Path
import yaml

from amazon_review_alignment.config import load_config

# 加载并合并 full.yaml 与 rlhf_a100.yaml
config = load_config(str(CONFIG))
config.pop("_config_path", None)

config["training"]["dpo"].update(
    {
        "learning_rate": 1e-5,
        "epochs": 1,
        "output_dir": (
            "outputs/a100-qwen3.5-2b/models/dpo-v2"
        ),
    }
)

DPO_V2_CONFIG = Path("/content/rlhf_a100_dpo_v2.yaml")
DPO_V2_CONFIG.write_text(
    yaml.safe_dump(
        config,
        sort_keys=False,
        allow_unicode=True,
    ),
    encoding="utf-8",
)

print("DPO v2 config:", DPO_V2_CONFIG)
print("Learning rate:", config["training"]["dpo"]["learning_rate"])
print("Epochs:", config["training"]["dpo"]["epochs"])
print("Output:", config["training"]["dpo"]["output_dir"])

In [ ]:
cli(
    "train-dpo",
    "--config",
    str(DPO_V2_CONFIG),
)

In [ ]:
from pathlib import Path
import shutil

prediction_dir = (
    REPO_DIR
    / "outputs"
    / "a100-qwen3.5-2b"
    / "evaluation"
    / "predictions"
)

old_prediction = prediction_dir / "dpo.jsonl"
backup_prediction = prediction_dir / "dpo-v1.jsonl"

if old_prediction.exists() and not backup_prediction.exists():
    shutil.copy2(old_prediction, backup_prediction)
    print("Backed up:", backup_prediction)

cli(
    "inference",
    "--config",
    str(DPO_V2_CONFIG),
    "--variant",
    "dpo",
    "--force",
)

## 7. 构建纯 RLAIF 数据

A100 正式流程不要求人工填写 200 条 A/B。Reward Model、PPO 和 GRPO
直接使用 OpenAI 教师生成并通过规则校验的 chosen/rejected 偏好。

这属于 RLAIF，而不是纯 RLHF。独立的 200 条人工盲评仅用于最终评估，
不进入训练数据。


In [ ]:
cli("build-rlhf-data", "--config", CONFIG)

import json

manifest_path = output_root / "rlhf" / "data_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))
assert manifest["alignment_method"] == "rlaif"
assert manifest["human_total_rows"] == 0


## 8. Reward Model、PPO 与 GRPO

每个阶段是独立单元格。阶段失败时先处理报错，不要跳过并继续。


In [ ]:
cli("train-reward", "--config", CONFIG)


In [ ]:
cli("train-ppo", "--config", CONFIG)


In [ ]:
cli("train-grpo", "--config", CONFIG)


## 9. 五模型统一评估和报告


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import pandas as pd

REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
os.chdir(REPO_DIR)

tracked_config = REPO_DIR / "configs" / "rlhf_a100_dpo_v2.yaml"
runtime_config = Path("/content/rlhf_a100_dpo_v2_runtime.yaml")

CONFIG_V2 = tracked_config if tracked_config.exists() else runtime_config

if not CONFIG_V2.exists():
    raise RuntimeError("找不到 DPO v2 配置文件")

def cli(*arguments):
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(arg) for arg in arguments),
    ]
    print("\n$", " ".join(command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, check=True)

output_root = REPO_DIR / "outputs" / "a100-qwen3.5-2b"
prediction_dir = output_root / "evaluation" / "predictions"

dpo_current = prediction_dir / "dpo.jsonl"
dpo_v2 = prediction_dir / "dpo-v2.jsonl"
grpo_prediction = prediction_dir / "grpo.jsonl"

if not dpo_current.exists():
    raise RuntimeError("dpo.jsonl 不存在")

dpo_rows = sum(1 for _ in dpo_current.open(encoding="utf-8"))
if dpo_rows != 500:
    raise RuntimeError(f"DPO v2 预测不完整：{dpo_rows}/500")

# 将已经完成的DPO v2结果加上明确版本标记
shutil.copy2(dpo_current, dpo_v2)
print("DPO v2 已确认:", dpo_v2)
print("DPO v2 rows:", dpo_rows)

# 现在只运行缺失的GRPO推理
if not grpo_prediction.exists():
    cli(
        "inference",
        "--config",
        CONFIG_V2,
        "--variant",
        "grpo",
    )
else:
    grpo_rows = sum(
        1 for _ in grpo_prediction.open(encoding="utf-8")
    )
    print(f"复用 GRPO prediction: {grpo_rows} rows")

# 五模型评估，不重新推理已有模型
cli(
    "evaluate",
    "--config",
    CONFIG_V2,
    "--variants",
    "base",
    "sft",
    "dpo",
    "ppo",
    "grpo",
)

cli("build-report", "--config", CONFIG_V2)

metrics_path = output_root / "evaluation" / "metrics.csv"
display(pd.read_csv(metrics_path))
print("Report:", output_root / "report.md")

DPO v2 已确认: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/evaluation/predictions/dpo-v2.jsonl
DPO v2 rows: 500

$ /usr/bin/python3 -u -m amazon_review_alignment.cli inference --config /content/rlhf_a100_dpo_v2_runtime.yaml --variant grpo
